# Model recall testing: workflow for WildObs Image Management Platform

## Description
- This script can be used for benchmarking an ai species recognition model with a local dataset to get an independent assessment of Recall for a given location
- The purpose is to inform an appropriate validation workflow for species relevant to the monitoring

## Setup instructions:
1) Prepare the testing dataset. Organise camera trap images into folders by species. Suggest using same number of images of each species e.g. 1000.
2) Establish a new project in the WildObs WIMP for benchmarking purposes. Name the project based on the model that will be tested e.g. "Model benchmark testing: WildObs National". You will need to create a separate project for each model.
3) Define tags in the project based on the scientific names of the species you are testing. These need to match with the names used in the WIMP.
4) Configure the project to use the model you want to test
5) Add a deployment to the project for each species and upload the relevant images into each deployment. Use the tags created earlier to assign to the deployment so you know which species it is supposed to be.
6) Run the images through the AI species recognition model
7) Export the project data in Camtrap DP format
8) Extract the data export to a folder
9) Use the folder path as input to this script

In [32]:
import pandas as pd
import os
import re
from IPython.display import display, HTML
display(HTML("<style>.container { width:100% !important; }</style>"))
pd.set_option('display.max_columns', None)
pd.set_option('display.width', None)


# -------- USER INPUT --------

model_name = 'WildObs National'
camtrap_folder = r"C:\Users\colin.broughton\Downloads\model-benchmarking-wildobs-national-20260317045910"

#model_name = 'AWC135'
#camtrap_folder = r"C:\Users\colin.broughton\Downloads\model-benchmarking-awc-135-20260317043737"


# ----------------------------


# -----------------------------
# Load required tables
# -----------------------------
observations = pd.read_csv(os.path.join(camtrap_folder, "observations.csv"))
media = pd.read_csv(os.path.join(camtrap_folder, "media.csv"))
deployments = pd.read_csv(os.path.join(camtrap_folder, "deployments.csv"))


# -----------------------------
# Keep only required columns
# -----------------------------
observations = observations[["eventID", "deploymentID", "scientificName"]]

media = media[["mediaID", "mediaComments"]]

deployments = deployments[["deploymentID", "deploymentTags"]]


# -----------------------------
# Extract sequenceID from mediaComments
# -----------------------------
def extract_sequence(comment):

    if pd.isna(comment):
        return None

    match = re.search(r"sequenceID:([^\s]+)", str(comment))

    if match:
        return match.group(1)

    return None


media["sequenceID"] = media["mediaComments"].apply(extract_sequence)


# -----------------------------
# Join media → observations
# -----------------------------
merged = pd.merge(
    media,
    observations,
    left_on="sequenceID",
    right_on="eventID",
    how="left"
)


# -----------------------------
# Join observations → deployments
# -----------------------------
merged = pd.merge(
    merged,
    deployments,
    on="deploymentID",
    how="left"
)


# -----------------------------
# Normalize values (clean only, no mapping)
# -----------------------------
merged["true_species"] = merged["deploymentTags"].astype(str).str.strip().str.lower()
merged["pred_species"] = merged["scientificName"].astype(str).str.strip().str.lower()


# -----------------------------
# Remove rows with missing truth
# -----------------------------
merged = merged[merged["true_species"].notna()]


# -----------------------------
# Recall calculation (generic)
# -----------------------------
results = []

species_list = sorted(merged["true_species"].dropna().unique())

for species in species_list:

    subset = merged[merged["true_species"] == species]

    tp = (subset["pred_species"] == species).sum()
    fn = (subset["pred_species"] != species).sum()

    recall = tp / (tp + fn) if (tp + fn) > 0 else 0

    results.append({
        "species": species,
        "images": len(subset),
        "true_positives": tp,
        "false_negatives": fn,
        "recall": round(recall, 4)
    })


results_df = pd.DataFrame(results)


In [33]:
# Output the results

print("\n")
print(f"Model recall: {model_name}")

display(results_df)

print("Detection Counts")
display(merged["true_species"].value_counts().rename_axis("species").reset_index(name="count"))

print("Predicted Species Counts")
display(merged["pred_species"].value_counts().rename_axis("species").reset_index(name="count"))

conf_matrix = pd.crosstab(
    merged["true_species"],
    merged["pred_species"]
)

print("Confusion Matrix")
display(conf_matrix)



Model recall: WildObs National


,species,images,true_positives,false_negatives,recall
0,felis catus,1014,840,174,0.8284
1,vulpes vulpes,1020,761,259,0.7461


Detection Counts


,species,count
0,vulpes vulpes,1020
1,felis catus,1014


Predicted Species Counts


,species,count
0,felis catus,876
1,vulpes vulpes,773
2,nan,58
3,canis lupus dingo,56
4,mammalia,51
5,carnivora,29
6,homo sapiens,26
7,canidae,25
8,perameles nasuta,18
9,wallabia bicolor,17


Confusion Matrix


pred_species,antechinus agilis,bos taurus,burhinus grallarius,canidae,canis familiaris,canis lupus dingo,canis lupus familiaris,carnivora,colluricincla harmonica,corcorax melanorhamphos,dama dama,felis,felis catus,geopelia humeralis,gymnorhina tibicen,heteromyias cinereifrons,homo sapiens,isoodon macrourus,mammalia,nan,notamacropus rufogriseus,oryctolagus cuniculus,passeriformes,perameles nasuta,sus scrofa,trichosurus caninus,trichosurus vulpecula,uromys caudimaculatus,varanus varius,vulpes vulpes,wallabia,wallabia bicolor
true_species,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,
felis catus,2,2,5,1,1,5,5,14,0,4,1,2,840,1,1,1,16,1,28,36,2,8,2,9,0,3,6,1,1,12,0,4
vulpes vulpes,0,1,0,24,13,51,9,15,1,0,1,5,36,0,0,0,10,0,23,22,3,4,0,9,4,1,9,3,0,761,2,13
